# Baseline Classifiers — Original Spambase Dataset (No SMOTE)

Uses the raw UCI Spambase dataset (`spambase.data`, 4,601 emails) with **no SMOTE** and **no feature engineering**.  
80/20 stratified split → **3,680 train / 921 test** at the natural 39.4/60.6% class distribution.  
These are the correct baseline results for the paper's Section IV.A and IV.B (pure unscaled + standardised baseline).  

This notebook replaces the baseline section that was accidentally run on SMOTE-balanced data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')
os.makedirs('baseline_figures', exist_ok=True)

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score,
    matthews_corrcoef, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

SEED = 0
np.random.seed(SEED)
print('Libraries loaded.')

## 1. Load Original Spambase Data

In [ ]:
# Load raw UCI spambase (no headers — 57 features + 1 label)
col_names = (
    [f'word_freq_{w}' for w in [
        'make','address','all','3d','our','over','remove','internet','order','mail',
        'receive','will','people','report','addresses','free','business','email','you',
        'credit','your','font','000','money','hp','hpl','george','650','lab','labs',
        'telnet','857','data','415','85','technology','1999','parts','pm','direct',
        'cs','meeting','original','project','re','edu','table','conference']] +
    [f'char_freq_{c}' for c in ['semicolon','lparen','lbracket','exclaim','dollar','hash']] +
    ['capital_run_length_average','capital_run_length_longest','capital_run_length_total',
     'spam']
)

df = pd.read_csv('spambase.data', header=None, names=col_names)

# Rename char_freq columns to match paper convention
df.rename(columns={
    'char_freq_semicolon': 'char_freq_;',
    'char_freq_lparen':    'char_freq_(',
    'char_freq_lbracket':  'char_freq_[',
    'char_freq_exclaim':   'char_freq_!',
    'char_freq_dollar':    'char_freq_$',
    'char_freq_hash':      'char_freq_#',
}, inplace=True)

X = df.drop('spam', axis=1).values
y = df['spam'].values
FEATURE_NAMES = df.drop('spam', axis=1).columns.tolist()

print(f'Dataset: {X.shape[0]} rows, {X.shape[1]} features')
print(f'Class distribution: Ham={int((y==0).sum())} ({(y==0).mean():.1%}), '
      f'Spam={int((y==1).sum())} ({(y==1).mean():.1%})')
print(f'Missing values: {np.isnan(X).sum()}')

In [ ]:
# 80/20 stratified split — natural class distribution preserved
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train spam: {y_train.mean():.1%}  |  Test spam: {y_test.mean():.1%}')

## 2. Model Definitions (identical params to paper)

In [ ]:
MODELS = {
    'XGBoost'            : XGBClassifier(eval_metric='logloss', use_label_encoder=False,
                                          n_jobs=-1, verbosity=0, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    'Gradient Boosting'  : GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                                       learning_rate=0.1, random_state=42),
    'SVM'                : SVC(C=10, kernel='rbf', probability=True, random_state=42),
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Naive Bayes'        : GaussianNB(),
    'Decision Tree'      : DecisionTreeClassifier(max_depth=10, random_state=42),
}
print(f'Defined {len(MODELS)} classifiers.')

## 3a. Pure Unscaled Baseline (no StandardScaler)

In [ ]:
def eval_model(name, model, Xtr, ytr, Xte, yte):
    model.fit(Xtr, ytr)
    yp = model.predict(Xte)
    yprob = model.predict_proba(Xte)[:,1] if hasattr(model,'predict_proba') else None
    return {
        'Model'   : name,
        'Accuracy': accuracy_score(yte, yp),
        'F1'      : f1_score(yte, yp),
        'AUC'     : roc_auc_score(yte, yprob) if yprob is not None else np.nan,
        'MCC'     : matthews_corrcoef(yte, yp),
        'Precision': precision_score(yte, yp),
        'Recall'  : recall_score(yte, yp),
        'FP'      : int(confusion_matrix(yte,yp)[0,1]),
        'FN'      : int(confusion_matrix(yte,yp)[1,0]),
    }

print('=== PURE UNSCALED BASELINE (original data, 3680/921 split) ===')
print('-'*80)
raw_results = []
for name, model in MODELS.items():
    r = eval_model(name, model, X_train, y_train, X_test, y_test)
    raw_results.append(r)
    print(f"{name:<22} Acc={r['Accuracy']:.4f}  F1={r['F1']:.4f}  "
          f"AUC={r['AUC']:.4f}  MCC={r['MCC']:.4f}  FP={r['FP']:3d}  FN={r['FN']:3d}")

df_raw = pd.DataFrame(raw_results).sort_values('Accuracy', ascending=False)
df_raw.to_csv('baseline_original_unscaled.csv', index=False)
print('\nSaved → baseline_original_unscaled.csv')

## 3b. Standardised Baseline (with StandardScaler)

In [ ]:
import copy
print('=== STANDARDISED BASELINE (original data, 3680/921 split) ===')
print('-'*80)
scaled_results = []
for name, model in MODELS.items():
    r = eval_model(name, copy.deepcopy(model), X_train_s, y_train, X_test_s, y_test)
    scaled_results.append(r)
    print(f"{name:<22} Acc={r['Accuracy']:.4f}  F1={r['F1']:.4f}  "
          f"AUC={r['AUC']:.4f}  MCC={r['MCC']:.4f}  FP={r['FP']:3d}  FN={r['FN']:3d}")

df_scaled = pd.DataFrame(scaled_results).sort_values('Accuracy', ascending=False)
df_scaled.to_csv('baseline_original_scaled.csv', index=False)
print('\nSaved → baseline_original_scaled.csv')

## 4. 10-Fold Cross-Validation (training set only)

In [ ]:
import copy
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scoring = ['accuracy', 'f1', 'roc_auc']

print('=== 10-FOLD CV (training set, original data) ===')
print('-'*80)
cv_results = {}
for name, model in MODELS.items():
    scores = cross_validate(copy.deepcopy(model), X_train_s, y_train,
                            cv=cv, scoring=cv_scoring, n_jobs=-1)
    cv_results[name] = {
        'acc_mean': scores['test_accuracy'].mean(),
        'acc_std' : scores['test_accuracy'].std(),
        'f1_mean' : scores['test_f1'].mean(),
        'auc_mean': scores['test_roc_auc'].mean(),
    }
    r = cv_results[name]
    print(f"{name:<22} CV Acc={r['acc_mean']:.4f}±{r['acc_std']:.4f}  "
          f"F1={r['f1_mean']:.4f}  AUC={r['auc_mean']:.4f}")

## 5. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = [
    ('Accuracy',  df_scaled.sort_values('Accuracy')),
    ('F1',        df_scaled.sort_values('F1')),
    ('AUC',       df_scaled.sort_values('AUC')),
]
for ax, (metric, df_sorted) in zip(axes, metrics):
    bars = ax.barh(df_sorted['Model'], df_sorted[metric],
                   color='steelblue', edgecolor='black', linewidth=0.5)
    for bar, val in zip(bars, df_sorted[metric]):
        ax.text(bar.get_width()+0.003, bar.get_y()+bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8)
    ax.set_xlabel(metric)
    ax.set_title(f'{metric} (Standardised, Original Data)')
    ax.set_xlim(0, 1.08)

plt.suptitle('Standardised Baseline — Original Spambase (3,680/921 split)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('baseline_figures/baseline_original_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 6. Paper-Ready Summary Table

In [ ]:
print('\n=== PAPER TABLE — Standardised Baseline (Original Data, 921-sample test) ===')
print(f'{"Classifier":<22} {"Acc":>7} {"F1":>7} {"AUC":>7} {"MCC":>7} {"FP":>4} {"FN":>4}')
print('-'*65)
for _, row in df_scaled.iterrows():
    print(f"{row['Model']:<22} {row['Accuracy']:>7.4f} {row['F1']:>7.4f} "
          f"{row['AUC']:>7.4f} {row['MCC']:>7.4f} {int(row['FP']):>4d} {int(row['FN']):>4d}")

print('\n=== PURE UNSCALED (XGB only — best unscaled) ===')
xgb_raw = [r for r in raw_results if r['Model'] == 'XGBoost'][0]
print(f"XGBoost (unscaled)     Acc={xgb_raw['Accuracy']:.4f}  "
      f"F1={xgb_raw['F1']:.4f}  AUC={xgb_raw['AUC']:.4f}  MCC={xgb_raw['MCC']:.4f}")

print('\n=== SVM WITHOUT SCALING (to show impact) ===')
svm_raw = [r for r in raw_results if r['Model'] == 'SVM'][0]
svm_scaled = [r for r in scaled_results if r['Model'] == 'SVM'][0]
delta = (svm_scaled['Accuracy'] - svm_raw['Accuracy'])*100
print(f"SVM unscaled: {svm_raw['Accuracy']:.4f}  →  SVM scaled: {svm_scaled['Accuracy']:.4f}"
      f"  (Δ={delta:+.2f} pp)")